In [12]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import numpy as np

In [8]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=""
)

if connection.is_connected():
    print("Conexión establecida correctamente")

Conexión establecida correctamente


In [9]:
cursor = connection.cursor()

cursor.execute("""
    CREATE DATABASE IF NOT EXISTS prueba_ladorian
""")

print("Base de datos creada correctamente")

Base de datos creada correctamente


In [10]:
connection.database = "prueba_ladorian"

In [11]:
create_table_query = """
CREATE TABLE IF NOT EXISTS ventas (
    site_id INT NOT NULL,
    date DATE NOT NULL,
    hour INT,
    product_id INT,
    category1_id INT,
    total DECIMAL(12,2),
    units DECIMAL(12,2),
    site_name VARCHAR(255),
    is_holidays BOOLEAN
);
"""

cursor.execute(create_table_query)

print("Tabla ventas creada correctamente")

Tabla ventas creada correctamente


In [19]:
df_sql = pd.read_csv("ventas_limpias.csv")

df_sql.head()

,site_id,date,product_id,category1_id,hour,units,total,site_name,is_holidays
0,60060,2022-03-03,2010029,119,8,1.0,4.70,Tienda Norte,0
1,60060,2022-02-06,2010029,119,23,2.0,9.40,Tienda Norte,0
2,60060,2022-02-21,2010520,119,20,1.0,4.90,Tienda Norte,0
3,60970,2022-01-20,3140194,119,11,1.0,4.45,Tienda Sur,0
4,60060,2022-12-25,2010029,119,9,1.0,5.05,Tienda Norte,0


In [20]:
df_sql.info()

<class 'pandas.DataFrame'>
RangeIndex: 75817 entries, 0 to 75816
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   site_id       75817 non-null  int64  
 1   date          75817 non-null  str    
 2   product_id    75817 non-null  int64  
 3   category1_id  75817 non-null  int64  
 4   hour          75817 non-null  int64  
 5   units         75817 non-null  float64
 6   total         75817 non-null  float64
 7   site_name     75817 non-null  str    
 8   is_holidays   75817 non-null  int64  
dtypes: float64(2), int64(5), str(2)
memory usage: 5.2 MB


In [21]:
df_sql["date"] = pd.to_datetime(
    df_sql["date"]
).dt.date

In [22]:
insert_query = """
INSERT INTO ventas (
    site_id,
    date,
    hour,
    product_id,
    category1_id,
    total,
    units,
    site_name,
    is_holidays
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [23]:
data = [
    tuple(row)
    for row in df_sql[
        [
            "site_id",
            "date",
            "hour",
            "product_id",
            "category1_id",
            "total",
            "units",
            "site_name",
            "is_holidays"
        ]
    ].itertuples(index=False, name=None)
]

In [24]:
cursor.executemany(insert_query, data)

connection.commit()

print(f"{cursor.rowcount} registros insertados correctamente")

75817 registros insertados correctamente


In [25]:
#Comprobar que realmente se han cargado
cursor.execute("SELECT COUNT(*) FROM ventas")

total_registros = cursor.fetchone()[0]

print(f"Registros en MySQL: {total_registros}")

Registros en MySQL: 75817


In [26]:
cursor.close()
connection.close()

print("Conexión cerrada")

Conexión cerrada
